In [1]:
2

2

In [5]:
from experiment_pipeline.config import GRANULARITIES
# from experiment_pipeline.step1_load_data import get_project_names
import json

with open("./experiment_pipeline/llm_inference_results.json", "r", encoding="utf-8") as file:
    data = json.load(file)

PROJECT_NAMES = data[GRANULARITIES[0]].keys()


invalid_counts = 0
for granularity in data.keys():
    granularity = data[granularity]
    for project_name in granularity.keys():
        project = granularity[project_name] 
        for i in project.keys():
            predicted_label = project[i]["predicted_label"]
            if "yes" in predicted_label.lower():
                predicted_label = 1
            elif "no" in predicted_label.lower():
                predicted_label = 0
            else:
                predicted_label = None
                invalid_counts += 1

            project[i]["predicted_label"] = predicted_label
#             data[granularity][project_name][i]["predicted_label"] = 0 if "no" in 

In [ ]:
import numpy as np
import sklearn.metrics as metrics

def compute_metrics(true_labels, pred_labels):
    """Computes Precision, Recall, F1, AUC, and MCC."""
    if len(true_labels) == 0:
        return [0.0] * 5
    
    p = metrics.precision_score(true_labels, pred_labels, zero_division=0)
    r = metrics.recall_score(true_labels, pred_labels, zero_division=0)
    f1 = metrics.f1_score(true_labels, pred_labels, zero_division=0)
    
    # Handle cases where only one class is present in true_labels for ROC AUC
    try:
        auc = metrics.roc_auc_score(true_labels, pred_labels)
    except ValueError:
        auc = 0.5
        
    mcc = metrics.matthews_corrcoef(true_labels, pred_labels)
    return [p, r, f1, auc, mcc]

def generate_latex_table(data_granite, data_litem):
    granularities = ["file", "class", "method", "block"]
    techniques = ["granite", "LiteM"]
    
    # Dynamically find all projects from the data
    projects = sorted(list(data_granite["file"].keys()))
    
    # 1. Compute and store metrics for all projects & techniques upfront
    # Structure: metrics_data[project][technique][granularity] -> list of 5 metrics
    metrics_data = {proj: {tech: {} for tech in techniques} for proj in projects}
    
    for proj in projects:
        for tech_name, data_dict in [("granite", data_granite), ("LiteM", data_litem)]:
            for gran in granularities:
                proj_gran_data = data_dict.get(gran, {}).get(proj, {})
                
                true_labels = []
                pred_labels = []
                
                sorted_keys = sorted(proj_gran_data.keys(), key=lambda x: int(x) if x.isdigit() else x)
                for idx in sorted_keys:
                    true_labels.append(proj_gran_data[idx]["true_label"])
                    pred_labels.append(proj_gran_data[idx]["predicted_label"])
                
                metrics_data[proj][tech_name][gran] = compute_metrics(true_labels, pred_labels)
                
    # 2. Compute Averages across all projects
    # Structure: averages[technique][granularity] -> list of 5 averaged metrics
    averages = {tech: {gran: [0.0] * 5 for gran in granularities} for tech in techniques}
    for tech in techniques:
        for gran in granularities:
            project_metrics = [metrics_data[proj][tech][gran] for proj in projects]
            if project_metrics:
                averages[tech][gran] = np.mean(project_metrics, axis=0).tolist()

    # Helper function to format strings to .2f and bold the highest value
    def format_val(val, max_val):
        val_str = f"{val:.2f}"
        # Use an epsilon window to capture absolute floats or ties safely
        if abs(val - max_val) < 1e-9:
            return f"\\textbf{{{val_str}}}"
        return val_str

    # 3. Construct Table Lines
    latex_lines = []
    latex_lines.append(r"\begin{table*}[t]")
    latex_lines.append(r"\centering")
    latex_lines.append(r"\caption{Evaluation Results Across Different Granularities.}")
    latex_lines.append(r"\label{tab:evaluation_results}")
    latex_lines.append(r"\resizebox{\textwidth}{!}{% Resize table to fit within text width if necessary")
    latex_lines.append(r"\begin{tabular}{ll" + "|ccccc" * (len(granularities)) + "}")
    latex_lines.append(r"\toprule")
    
    # Header Row 1
    header1 = r"\multirow{2}{*}{Project} & \multirow{2}{*}{Technique}"
    for gran in granularities:
        header1 += f" & \\multicolumn{{5}}{{c}}{{{gran.capitalize()}}}"
    header1 += r" \\"
    latex_lines.append(header1)
    
    # Midrules for each granularity group
    midrules = r"\cmidrule(lr){3-7} \cmidrule(lr){8-12} \cmidrule(lr){13-17} \cmidrule(lr){18-22}"
    latex_lines.append(midrules)
    
    # Header Row 2
    header2 = r" & & " + " & ".join(["P & R & F1 & AUC & MCC"] * len(granularities)) + r" \\"
    latex_lines.append(header2)
    latex_lines.append(r"\midrule")
    
    # Populate project rows
    for proj in projects:
        for tech_name in techniques:
            row_str = ""
            if tech_name == "granite":
                row_str += f"\\multirow{{2}}{{*}}{{{proj}}}"
            else:
                row_str += " "
                
            row_str += f" & {tech_name}"
            
            for gran in granularities:
                vals = metrics_data[proj][tech_name][gran]
                other_tech = "LiteM" if tech_name == "granite" else "granite"
                other_vals = metrics_data[proj][other_tech][gran]
                
                for m_idx in range(5):
                    max_val = max(vals[m_idx], other_vals[m_idx])
                    row_str += f" & {format_val(vals[m_idx], max_val)}"
            
            row_str += r" \\"
            latex_lines.append(row_str)
            
        if proj != projects[-1]:
            latex_lines.append(r"\hline")
        
    # Populate Summary Average Rows at the bottom
    latex_lines.append(r"\midrule")
    for tech_name in techniques:
        row_str = ""
        if tech_name == "granite":
            row_str += f"\\multirow{{2}}{{*}}{{Average}}"
        else:
            row_str += " "
            
        row_str += f" & {tech_name}"
        
        for gran in granularities:
            vals = averages[tech_name][gran]
            other_tech = "LiteM" if tech_name == "granite" else "granite"
            other_vals = averages[other_tech][gran]
            
            for m_idx in range(5):
                max_val = max(vals[m_idx], other_vals[m_idx])
                row_str += f" & {format_val(vals[m_idx], max_val)}"
                
        row_str += r" \\"
        latex_lines.append(row_str)
        
    latex_lines.append(r"\bottomrule")
    latex_lines.append(r"\end{tabular}")
    latex_lines.append(r"}")
    latex_lines.append(r"\end{table*}")
    
    return "\n".join(latex_lines)

In [21]:
data_granite = data
data_litem = data

print(generate_latex_table(data_granite=data, data_litem=data))

\begin{table*}[t]
\centering
\caption{Evaluation Results Across Different Granularities.}
\label{tab:evaluation_results}
\resizebox{\textwidth}{!}{% Resize table to fit within text width if necessary
\begin{tabular}{ll|ccccc|ccccc|ccccc|ccccc}
\toprule
\multirow{2}{*}{Project} & \multirow{2}{*}{Technique} & \multicolumn{5}{c}{File} & \multicolumn{5}{c}{Class} & \multicolumn{5}{c}{Method} & \multicolumn{5}{c}{Block} \\
\cmidrule(lr){3-7} \cmidrule(lr){8-12} \cmidrule(lr){13-17} \cmidrule(lr){18-22}
 & & P & R & F1 & AUC & MCC & P & R & F1 & AUC & MCC & P & R & F1 & AUC & MCC & P & R & F1 & AUC & MCC \\
\midrule
\multirow{2}{*}{antlr4} & granite & 0.12 & 1.00 & 0.21 & 0.50 & 0.00 & 0.15 & 1.00 & 0.26 & 0.50 & 0.00 & 0.04 & 1.00 & 0.08 & 0.50 & 0.00 & 0.02 & 1.00 & 0.05 & 0.50 & 0.00 \\
  & LiteM & 0.12 & 1.00 & 0.21 & 0.50 & 0.00 & 0.15 & 1.00 & 0.26 & 0.50 & 0.00 & 0.04 & 1.00 & 0.08 & 0.50 & 0.00 & 0.02 & 1.00 & 0.05 & 0.50 & 0.00 \\
\hline
\multirow{2}{*}{guava} & granite & 0.31 & 1.0